# OpenSearch Serverless vs S3 Vectors: Comparison

이 노트북에서는 02와 03에서 측정한 벤치마크 결과를 시각화하여 두 서비스를 비교합니다.

비교 항목:
1. **인제스트 속도** — 동일한 벡터를 저장하는 데 걸리는 시간
2. **검색 레이턴시** — k값별 p50/p95/p99 비교
3. **기능 비교** — 하이브리드 검색, 메타데이터 필터링, 비용 모델 등
4. **워크로드별 선택 가이드**

**사전 조건:** `02_opensearch_serverless.ipynb`와 `03_s3_vectors.ipynb` 실행 완료

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

with open("benchmark_opensearch.json") as f:
    os_data = json.load(f)
with open("benchmark_s3vectors.json") as f:
    s3v_data = json.load(f)

print(f"✅ 벤치마크 데이터 로드 완료")
print(f"   OpenSearch: {os_data['vector_count']}개 벡터, 인제스트 {os_data['ingest_time']:.2f}s")
print(f"   S3 Vectors: {s3v_data['vector_count']}개 벡터, 인제스트 {s3v_data['ingest_time']:.2f}s")

## 1. Ingest Speed Comparison

동일한 벡터 데이터를 각 서비스에 저장하는 데 걸린 시간을 비교합니다.

> 📖 **배치 크기 차이**: OpenSearch는 bulk API로 배치 크기 제한이 유연하고(기본 100MB), S3 Vectors는 요청당 최대 500개입니다. 각 서비스의 API 특성에 맞는 최적 배치 크기를 사용했습니다.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
services = ['OpenSearch\nServerless', 'S3 Vectors']
times = [os_data['ingest_time'], s3v_data['ingest_time']]
colors = ['#FF6B35', '#2196F3']

bars = ax.bar(services, times, color=colors, width=0.5, edgecolor='white', linewidth=1.5)
for bar, t, bs in zip(bars, times, [os_data['ingest_batch_size'], s3v_data['ingest_batch_size']]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{t:.2f}s\n(batch={bs})', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Time (seconds)', fontsize=12)
ax.set_title(f'Ingest Speed — {os_data["vector_count"]} vectors', fontsize=14, fontweight='bold')
ax.set_ylim(0, max(times) * 1.4)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 2. Search Latency Comparison

k값(5, 10, 20)별 검색 레이턴시를 비교합니다. p50(중앙값)과 p95(95번째 퍼센타일)를 함께 표시합니다.

In [ ]:
k_values = ['k=5', 'k=10', 'k=20']
x = np.arange(len(k_values))
width = 0.35

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# p50 comparison
os_p50 = [os_data['search'][k]['p50'] for k in k_values]
s3v_p50 = [s3v_data['search'][k]['p50'] for k in k_values]

bars1 = ax1.bar(x - width/2, os_p50, width, label='OpenSearch', color='#FF6B35', edgecolor='white')
bars2 = ax1.bar(x + width/2, s3v_p50, width, label='S3 Vectors', color='#2196F3', edgecolor='white')
for bars in [bars1, bars2]:
    for bar in bars:
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9)
ax1.set_xlabel('k value'); ax1.set_ylabel('Latency (ms)')
ax1.set_title('p50 (Median) Latency', fontsize=13, fontweight='bold')
ax1.set_xticks(x); ax1.set_xticklabels(k_values)
ax1.legend(); ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

# p95 comparison
os_p95 = [os_data['search'][k]['p95'] for k in k_values]
s3v_p95 = [s3v_data['search'][k]['p95'] for k in k_values]

bars1 = ax2.bar(x - width/2, os_p95, width, label='OpenSearch', color='#FF6B35', edgecolor='white')
bars2 = ax2.bar(x + width/2, s3v_p95, width, label='S3 Vectors', color='#2196F3', edgecolor='white')
for bars in [bars1, bars2]:
    for bar in bars:
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9)
ax2.set_xlabel('k value'); ax2.set_ylabel('Latency (ms)')
ax2.set_title('p95 (Tail) Latency', fontsize=13, fontweight='bold')
ax2.set_xticks(x); ax2.set_xticklabels(k_values)
ax2.legend(); ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

plt.suptitle(f'Search Latency — {os_data["vector_count"]} vectors', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Latency summary table
print(f"{'Service':<25} {'k=5 p50':>10} {'k=5 p95':>10} {'k=10 p50':>10} {'k=20 p50':>10}")
print('-' * 70)
for data, name in [(os_data, 'OpenSearch Serverless'), (s3v_data, 'S3 Vectors')]:
    print(f"{name:<25} {data['search']['k=5']['p50']:>9.1f}ms {data['search']['k=5']['p95']:>9.1f}ms "
          f"{data['search']['k=10']['p50']:>9.1f}ms {data['search']['k=20']['p50']:>9.1f}ms")

## 3. Workload Selection Guide

두 서비스는 서로 다른 설계 철학을 가지고 있어, 워크로드 특성에 따라 적합한 선택이 달라집니다.

### OpenSearch Serverless를 선택하는 경우
- 비디오 메타데이터(제목, 설명, 태그)에 대한 **키워드 검색**과 임베딩 기반 **시맨틱 검색**을 동시에 수행해야 할 때
- **하이브리드 검색**으로 더 정확한 랭킹이 필요할 때
- 검색 레이턴시가 **25ms 이하**로 중요한 실시간 서비스
- HNSW, IVF 등 **인덱스 알고리즘을 직접 튜닝**하고 싶을 때

### S3 Vectors를 선택하는 경우
- **빠르게 시작**하고 싶을 때 (Vector Bucket + Index 생성만으로 완료)
- **비용 효율성**이 중요한 소규모 워크로드 (OCU 최소 비용 없음)
- 벡터 **upsert**(재생성 시 자동 업데이트)가 필요할 때
- 기존 **S3 인프라**와 자연스럽게 통합하고 싶을 때
- 풀텍스트 검색 없이 **순수 벡터 유사도 검색**만 필요할 때

## Summary

이 워크샵에서는 TwelveLabs Marengo 3.0으로 생성한 비디오 임베딩을 AWS의 두 가지 벡터 저장소에 저장하고 검색하는 과정을 실습했습니다.

**핵심 요약:**
- **OpenSearch Serverless** — 하이브리드 검색(벡터 + 키워드)이 필요하고, 낮은 검색 레이턴시가 중요한 경우
- **S3 Vectors** — 간편한 설정과 비용 효율성이 중요하고, 순수 벡터 검색만 필요한 경우

실제 프로덕션 도입 시에는 자신의 워크로드 데이터와 쿼리 패턴으로 PoC를 수행하는 것을 권장합니다. 이 워크샵의 벡터 수(수백~수천 개)는 소규모 테스트이며, 대규모(수십만~수백만 벡터)에서는 각 서비스의 스케일링 특성에 따라 결과가 달라질 수 있습니다.